# D18 — Reavaliação do gate pré-estimação

## tl;dr

A D18 separa prontidão técnica de suficiência de identificação. A amostra D15 permanece tecnicamente íntegra e implementável; os diagnósticos de identificação exigem ressalvas, mas não demonstram um bloqueador objetivo para começar a implementação futura do estimador. Nenhum efeito causal é estimado aqui.

- Os **5 tratados** fora do intervalo simétrico D17 estão todos **acima do máximo dos controles**; os **1.361 controles** fora estão todos abaixo do mínimo dos tratados. Só o primeiro caso é falta de suporte diretamente relevante para ATT.
- As grandes diferenças de slopes em nível são fortemente atenuadas por `log1p`, índice `g-1=100` e mudanças percentuais. Isso mostra sensibilidade à escala, não valida tendências paralelas.
- As primeiras diferenças ainda exibem heterogeneidade por coorte; 2011–2012 merecem ressalva, sem gate por p-valor.
- A influência de municípios grandes é material, mas não explica toda a divergência pré em 2011–2012.
- A coorte 2013 tem suporte no score para seus dois municípios; o problema principal é **PRECISAO**, com risco adicional de **TRAJETORIA**.
- Mesmo no cenário espacial mais severo (100 km), restariam **1.323 controles**.

**Gates:** `D18_PRONTO_TECNICAMENTE_PARA_IMPLEMENTAR_ESTIMADOR = SIM`; `D18_IDENTIFICACAO_SUFICIENTE_PARA_INTERPRETACAO_CAUSAL = COM_RESSALVAS`; `DESENHO_CAUSAL_APROVADO = NAO`.

## Contexto, método e fronteiras

A unidade é município-ano, 2007–2019. O outcome primário congelado é `pessoal_ocupado_assalariado` em nível; o tratamento, as coortes, o timing, o grupo never-treated e a hipótese `anticipation=0` permanecem inalterados. Toda a evidência desta etapa é pré-tratamento, estrutural ou metadado do desenho.

A leitura metodológica segue Callaway & Sant'Anna (2021), Goodman-Bacon (2021), Sun & Abraham (2021) e Roth (2022): diferenças de nível não são violações automáticas de tendências paralelas; ausência de rejeição de leads não comprova a hipótese; TWFE ingênuo não é estimador principal adequado; e overlap deve ser lido em relação ao estimando.

**Proibições respeitadas:** nenhum ATT, event-study causal, TWFE causal, Callaway–Sant'Anna real, matching, trimming, peso, exclusão de unidade/coorte ou uso de outcome pós-tratamento.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import diagnostica_pre_estimacao as d17
import reavalia_gate_pre_estimacao as d18
from visualizacao_ipt import aplicar_tema_ipt, salvar_figura_ipt, estilizar_tabela_ipt, CORES_IPT

PATH_AMOSTRA = ROOT / 'data' / 'processed' / 'amostra_causal_cempre_2007_2019.parquet'
PATH_DISTANCIAS = ROOT / 'data' / 'processed' / 'diagnostico_distancias_spillover_fase_ii.parquet'
PATH_ARRANJOS = ROOT / 'data' / 'processed' / 'diagnostico_arranjos_populacionais_fase_ii.parquet'
PATH_SCORES_D17 = ROOT / 'outputs' / 'diagnostics' / 'diagnostico_overlap.csv'
OUT_DIAG = ROOT / 'outputs' / 'diagnostics'
OUT_FIG = ROOT / 'outputs' / 'figures'
COVARIAVEIS = ['qt_unidades_locais', 'pessoal_ocupado_total', 'pessoal_ocupado_assalariado', 'salario_medio_reais_nominal']

def mostrar(df, n=30):
    display(estilizar_tabela_ipt(df.head(n)))

def salvar_figura(fig, nome):
    aplicar_tema_ipt(fig)
    caminho = salvar_figura_ipt(fig, OUT_FIG / f'{nome}.png')
    print(f'Figura preservada em: {caminho.relative_to(ROOT)}')
    return fig

def git(*args):
    return subprocess.run(['git', *args], cwd=ROOT, check=True, capture_output=True, text=True).stdout.strip()

## 1. Estado inicial, inputs e preservação

In [2]:
estado_git_inicial = {'branch': git('branch', '--show-current'), 'head': git('rev-parse', 'HEAD'), 'origin_main': git('rev-parse', 'origin/main'), 'status': git('status', '--short', '--', '.')}
display(pd.DataFrame({'campo': estado_git_inicial.keys(), 'valor': estado_git_inicial.values()}))
hash_d15_antes = d17.sha256_arquivo(PATH_AMOSTRA)
amostra = pd.read_parquet(PATH_AMOSTRA)
distancias = pd.read_parquet(PATH_DISTANCIAS)
arranjos = pd.read_parquet(PATH_ARRANJOS)
scores_d17 = pd.read_csv(PATH_SCORES_D17, dtype={'codigo_municipio_ibge': 'string'})
auditoria_input = d17.auditar_input_congelado(amostra)
mostrar(pd.DataFrame({'metrica': auditoria_input.keys(), 'valor': [str(v) for v in auditoria_input.values()]}), 30)

,campo,valor
0,branch,main
1,head,c59022fc15e081411ff09bdf8765bc6347d15b00
2,origin_main,c59022fc15e081411ff09bdf8765bc6347d15b00
3,status,M README.md\n M docs/playbooks/ESTADO_ATUAL.md...


,metrica,valor
0,n_tratados,129
1,n_controles,4963
2,n_unidades,5092
3,n_linhas,66196
4,n_linhas_elegiveis,66195
5,sobreposicao_tratado_controle,0
6,unidades_painel_desbalanceado,0
7,coortes,"{2009: 21, 2010: 27, 2011: 66, 2012: 13, 2013: 2}"
8,n_missing_outcome,0
9,status_outcome,"['observado', 'zero_real']"


## 2. Auditoria crítica da D17

A D17 mediu corretamente os objetos que declarou e preservou a amostra, mas seu gate final agregou riscos de naturezas diferentes. Três refinamentos são necessários: (1) imbalance em nível mede composição/escala, não tendência; (2) o intervalo comum simétrico mistura falta de suporte dos tratados com controles excedentes; (3) precisão, plausibilidade de spillover e ausência de covariáveis externas não são, isoladamente, falhas demonstradas de identificação. A evidência real não é suavizada: cinco tratados continuam além do máximo observado entre controles, 2011–2012 mantêm heterogeneidade pré e 2013 tem apenas dois tratados.

## 3. Overlap orientado ao ATT

In [3]:
baseline = d17.construir_baseline_comum(amostra, COVARIAVEIS, ano_base=2007)
scores_att, resumo_overlap_att = d18.classificar_suporte_att(scores_d17)
extremos_att = d18.nearest_support_descritivo(baseline, scores_att, COVARIAVEIS, k_vizinhos=5)
overlap_export = scores_att.merge(extremos_att, on=['codigo_municipio_ibge', 'propensity_score', 'status_suporte_att'], how='left', suffixes=('', '_extremo'), validate='one_to_one')
overlap_export.to_csv(OUT_DIAG / 'D18_overlap_att.csv', index=False, encoding='utf-8-sig')
mostrar(pd.DataFrame({'metrica': resumo_overlap_att.keys(), 'valor': resumo_overlap_att.values()}), 30)
mostrar(extremos_att, 10)

,metrica,valor
0,score_min_tratados,0.000834
1,score_max_tratados,0.962359
2,score_min_controles,0.000013
3,score_max_controles,0.848672
4,n_tratados_abaixo_min_controles,0.000000
5,n_tratados_acima_max_controles,5.000000
6,n_controles_abaixo_min_tratados,1361.000000
7,n_controles_acima_max_tratados,0.000000
8,n_tratados_dentro_faixa_controles,124.000000
9,n_controles_na_faixa_tratados,3602.000000


,codigo_municipio_ibge,municipio,uf,coorte_g,propensity_score,status_suporte_att,codigo_controle_mais_proximo,municipio_controle_mais_proximo,score_controle_mais_proximo,distancia_padronizada_mais_proximo,distancia_padronizada_media_k,k_vizinhos,rotulo_uso,tratado__qt_unidades_locais,controle_mais_proximo__qt_unidades_locais,tratado__pessoal_ocupado_total,controle_mais_proximo__pessoal_ocupado_total,tratado__pessoal_ocupado_assalariado,controle_mais_proximo__pessoal_ocupado_assalariado,tratado__salario_medio_reais_nominal,controle_mais_proximo__salario_medio_reais_nominal
0,2910800,Feira de Santana - BA,29,2012.000000,0.962359,TRATADO_ACIMA_MAX_CONTROLES,2914802,Itabuna - BA,0.838638,1.176636,1.566925,5,NEAREST_SUPPORT_DESCRITIVO__NAO_MATCHING_NAO_TRIMMING,12961.000000,5384.000000,91393.000000,39217.000000,75422.000000,31644.000000,782.700000,779.510000
1,2604106,Caruaru - PE,26,2011.000000,0.907542,TRATADO_ACIMA_MAX_CONTROLES,2914802,Itabuna - BA,0.838638,0.731094,1.473832,5,NEAREST_SUPPORT_DESCRITIVO__NAO_MATCHING_NAO_TRIMMING,6700.000000,5384.000000,53745.000000,39217.000000,45875.000000,31644.000000,650.020000,779.510000
2,5201108,Anápolis - GO,52,2011.000000,0.892030,TRATADO_ACIMA_MAX_CONTROLES,3167202,Sete Lagoas - MG,0.299496,1.513123,1.574039,5,NEAREST_SUPPORT_DESCRITIVO__NAO_MATCHING_NAO_TRIMMING,7822.000000,5566.000000,73116.000000,51137.000000,64531.000000,43725.000000,843.670000,900.410000
3,4305108,Caxias do Sul - RS,43,2011.000000,0.877900,TRATADO_ACIMA_MAX_CONTROLES,4115200,Maringá - PR,0.828910,1.145302,1.340686,5,NEAREST_SUPPORT_DESCRITIVO__NAO_MATCHING_NAO_TRIMMING,22424.000000,14590.000000,176571.000000,128865.000000,144161.000000,107309.000000,1341.130000,979.190000
4,1506807,Santarém - PA,15,2010.000000,0.874397,TRATADO_ACIMA_MAX_CONTROLES,4108403,Francisco Beltrão - PR,0.292362,1.464360,1.471795,5,NEAREST_SUPPORT_DESCRITIVO__NAO_MATCHING_NAO_TRIMMING,3540.000000,3340.000000,30911.000000,23186.000000,26959.000000,18512.000000,758.790000,777.400000


In [4]:
contagens_overlap = scores_att.groupby(['status_suporte_att', 'implicacao_att'], as_index=False).size()
fig = px.bar(contagens_overlap, x='size', y='status_suporte_att', orientation='h', color='implicacao_att', labels={'size': 'Municípios', 'status_suporte_att': 'Posição no suporte', 'implicacao_att': 'Implicação para ATT'}, title='Overlap diagnóstico: cauda de controles não equivale a falta de suporte do tratado')
fig.update_layout(legend={'orientation': 'h'})
salvar_figura(fig, 'd18_01_overlap_att')
fig.show()

Figura preservada em: outputs\figures\interactive\d18_01_overlap_att.html


**Resultado.** A D17 contou corretamente cinco tratados e 1.361 controles fora da interseção. A D18 refina a interpretação: os cinco tratados estão acima do máximo dos controles e constituem o problema relevante de suporte para ATT; os 1.361 controles estão abaixo do mínimo dos tratados e representam oferta excedente de controles, não falta de suporte do tratado. Nenhuma unidade é removida. A distância padronizada usa as quatro covariáveis CEMPRE de 2007 e macrorregião; é nearest-support descritivo, não matching.

## 4. Escala versus tendência

In [5]:
tendencias = d18.resumir_tendencias_escala(amostra)
tendencias.to_csv(OUT_DIAG / 'D18_pre_tendencias_escala.csv', index=False, encoding='utf-8-sig')
mostrar(tendencias, 10)
plot_tendencias = tendencias.melt(id_vars=['coorte_g', 'n_tratados'], value_vars=['diferenca_slope_nivel', 'diferenca_slope_log'], var_name='escala', value_name='diferenca')
fig = px.bar(plot_tendencias, x='coorte_g', y='diferenca', color='escala', facet_col='escala', facet_col_wrap=1, labels={'coorte_g': 'Coorte', 'diferenca': 'Tratados − controles', 'escala': 'Métrica'}, title='Diferença de slopes pré: nível e log1p não têm a mesma leitura', height=650)
fig.update_yaxes(matches=None)
salvar_figura(fig, 'd18_02_slopes_nivel_log')
fig.show()

,coorte_g,n_tratados,slope_nivel_tratados,slope_nivel_controles,diferenca_slope_nivel,slope_log_tratados,slope_log_controles,diferenca_slope_log,mudanca_percentual_tratados,mudanca_percentual_controles,diagnostico,uso_log1p,rotulo_inferencia
0,2009,21,379.142857,75.945396,303.197461,0.008047,0.016278,-0.008231,1.392753,6.639091,SINAL_DIVERGENCIA_MUDA_APOS_RETIRAR_ESCALA,APENAS_DIAGNOSTICO_NAO_ALTERA_OUTCOME_PRINCIPAL,DESCRITIVO_NAO_TESTE_DE_TENDENCIAS_PARALELAS
1,2010,27,209.240741,101.689301,107.551440,0.031529,0.058619,-0.027090,9.063148,20.222592,SINAL_DIVERGENCIA_MUDA_APOS_RETIRAR_ESCALA,APENAS_DIAGNOSTICO_NAO_ALTERA_OUTCOME_PRINCIPAL,DESCRITIVO_NAO_TESTE_DE_TENDENCIAS_PARALELAS
2,2011,66,1009.696970,131.506810,878.190159,0.080647,0.063218,0.017429,32.118837,29.166782,DIVERGENCIA_LOG_PEQUENA_MAS_NAO_TESTE_DE_PARALLEL_TRENDS,APENAS_DIAGNOSTICO_NAO_ALTERA_OUTCOME_PRINCIPAL,DESCRITIVO_NAO_TESTE_DE_TENDENCIAS_PARALELAS
3,2012,13,1637.430769,135.564920,1501.865849,0.063091,0.062809,0.000282,29.287132,36.394510,DIVERGENCIA_LOG_PEQUENA_MAS_NAO_TESTE_DE_PARALLEL_TRENDS,APENAS_DIAGNOSTICO_NAO_ALTERA_OUTCOME_PRINCIPAL,DESCRITIVO_NAO_TESTE_DE_TENDENCIAS_PARALELAS
4,2013,2,2103.771429,122.459054,1981.312374,0.091065,0.048597,0.042468,52.441637,33.728685,DIVERGENCIA_PERSISTE_EM_LOG_COM_MAGNITUDE_DESCRITIVA,APENAS_DIAGNOSTICO_NAO_ALTERA_OUTCOME_PRINCIPAL,DESCRITIVO_NAO_TESTE_DE_TENDENCIAS_PARALELAS


Figura preservada em: outputs\figures\interactive\d18_02_slopes_nivel_log.html


In [6]:
indices = []
for g in d18.COORTES:
    painel_g = d18.selecionar_painel_pre_coorte(amostra, g)
    indice_g = d18.normalizar_indice_g_menos_1(painel_g, g)
    indices.append(indice_g.groupby(['grupo', 'tempo_relativo_pre'], as_index=False).agg(media_indice=('indice_g_menos_1', 'mean'), mediana_indice=('indice_g_menos_1', 'median'), n=('codigo_municipio_ibge', 'nunique')).assign(coorte_g=g))
indices_pre = pd.concat(indices, ignore_index=True)
fig = px.line(indices_pre, x='tempo_relativo_pre', y='media_indice', color='grupo', facet_col='coorte_g', facet_col_wrap=2, markers=True, labels={'tempo_relativo_pre': 'k (somente pré)', 'media_indice': 'Índice médio (g−1 = 100)', 'grupo': 'Grupo'}, title='Trajetórias pré normalizadas por coorte', height=800)
fig.add_hline(y=100, line_dash='dash', line_color=CORES_IPT['AZUL_ESCURO'])
salvar_figura(fig, 'd18_03_indice_g_menos_1')
fig.show()

Figura preservada em: outputs\figures\interactive\d18_03_indice_g_menos_1.html


**Resultado.** A divergência em nível é muito sensível à escala. Em 2012, por exemplo, a diferença de slope cai de aproximadamente 1.501,9 pessoas/ano para 0,00028 em log; em 2009–2010 o sinal muda. Isso enfraquece a leitura de que slopes em nível, sozinhos, demonstrariam violação. A coorte 2013 mantém diferença log maior, porém com `n=2`. Nenhuma transformação é escolhida para a estimação futura.

## 5. Pré-tendência em primeiras diferenças

In [7]:
primeiras_diferencas = d18.resumir_primeiras_diferencas(amostra)
primeiras_diferencas.to_csv(OUT_DIAG / 'D18_primeiras_diferencas.csv', index=False, encoding='utf-8-sig')
mostrar(primeiras_diferencas, 10)
fig = px.bar(primeiras_diferencas, x='coorte_g', y='smd_primeiras_diferencas', text='smd_primeiras_diferencas', color_discrete_sequence=[CORES_IPT['AZUL_PRINCIPAL']], labels={'coorte_g': 'Coorte', 'smd_primeiras_diferencas': 'SMD de ΔY pré'}, title='Diferenças padronizadas das primeiras diferenças pré')
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
salvar_figura(fig, 'd18_04_primeiras_diferencas')
fig.show()

,coorte_g,n_deltas_tratados,n_deltas_controles,n_unidades_tratados,n_unidades_controles,media_tratados,mediana_tratados,p10_tratados,p25_tratados,p75_tratados,p90_tratados,media_controles,mediana_controles,p10_controles,p25_controles,p75_controles,p90_controles,diferenca_media,smd_primeiras_diferencas,erro_padrao_descritivo_por_unidade,ic95_descritivo_inferior,ic95_descritivo_superior,rotulo_inferencia
0,2009,21,4963,21,4963,379.142857,230.000000,-967.000000,14.000000,978.000000,1455.000000,75.945396,15.000000,-112.000000,-23.000000,89.000000,294.600000,303.197461,0.236961,309.890149,-304.187231,910.582154,INCERTEZA_DESCRITIVA_NAO_P_VALOR_DE_GATE
1,2010,53,9926,27,4963,314.584906,245.000000,-706.400000,-164.000000,1001.000000,1742.000000,101.689301,29.000000,-95.000000,-12.000000,114.000000,338.000000,212.895605,0.179767,268.027270,-312.437844,738.229054,INCERTEZA_DESCRITIVA_NAO_P_VALOR_DE_GATE
2,2011,198,14889,66,4963,1040.106061,485.500000,-124.700000,127.500000,1274.000000,2977.000000,131.959433,31.000000,-82.000000,-8.000000,124.000000,363.000000,908.146627,0.589535,178.455791,558.373277,1257.919978,INCERTEZA_DESCRITIVA_NAO_P_VALOR_DE_GATE
3,2012,52,19852,13,4963,1654.711538,813.000000,87.300000,289.000000,2667.000000,4593.000000,129.464538,32.000000,-75.000000,-6.000000,125.000000,367.000000,1525.247001,0.802355,615.373191,319.115547,2731.378455,INCERTEZA_DESCRITIVA_NAO_P_VALOR_DE_GATE
4,2013,10,24815,2,4963,1884.300000,839.500000,-1809.100000,159.750000,927.000000,10291.200000,110.713157,26.000000,-101.000000,-15.000000,112.000000,342.600000,1773.586843,0.515687,1075.525470,-334.443078,3881.616764,INCERTEZA_DESCRITIVA_NAO_P_VALOR_DE_GATE


Figura preservada em: outputs\figures\interactive\d18_04_primeiras_diferencas.html


As primeiras diferenças distinguem nível de evolução, mas continuam expressas na escala do outcome. Os SMDs pré variam por coorte e são maiores em 2011–2012. Os intervalos são incerteza descritiva da diferença média, não p-valores nem testes decisórios de parallel trends.

## 6. Influência de municípios grandes — somente pré-tratamento

In [8]:
influencia = d18.diagnostico_influencia_pre(amostra)
influencia.to_csv(OUT_DIAG / 'D18_influencia_pre.csv', index=False, encoding='utf-8-sig')
mostrar(influencia, 30)
fig = px.line(influencia, x='cenario', y='diferenca_slope_nivel', color='coorte_g', markers=True, labels={'cenario': 'Omissão apenas diagnóstica', 'diferenca_slope_nivel': 'Diferença de slope em nível', 'coorte_g': 'Coorte'}, title='Influência pré-tratamento dos maiores municípios')
fig.add_hline(y=0, line_dash='dash', line_color=CORES_IPT['AZUL_ESCURO'])
salvar_figura(fig, 'd18_05_influencia_pre')
fig.show()

,coorte_g,cenario,n_tratados_original,n_tratados_diagnostico,n_tratados_omitidos_apenas_diagnostico,slope_nivel_tratados,slope_nivel_controles,diferenca_slope_nivel,rotulo_uso,diferenca_sem_omissao,mudanca_diferenca_vs_sem_omissao
0,2009,SEM_OMISSAO,21,21,0,379.142857,75.945396,303.197461,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,303.197461,0.000000
1,2009,OMITE_MAIOR_1,21,21,0,379.142857,75.945396,303.197461,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,303.197461,0.000000
2,2009,OMITE_TOP_5,21,20,1,215.000000,75.945396,139.054604,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,303.197461,-164.142857
3,2009,OMITE_TOP_10,21,19,2,164.473684,75.945396,88.528288,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,303.197461,-214.669173
4,2010,SEM_OMISSAO,27,27,0,209.240741,101.689301,107.551440,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,107.551440,0.000000
5,2010,OMITE_MAIOR_1,27,27,0,209.240741,101.689301,107.551440,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,107.551440,0.000000
6,2010,OMITE_TOP_5,27,27,0,209.240741,101.689301,107.551440,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,107.551440,0.000000
7,2010,OMITE_TOP_10,27,26,1,152.346154,101.689301,50.656853,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,107.551440,-56.894587
8,2011,SEM_OMISSAO,66,66,0,1009.696970,131.506810,878.190159,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,878.190159,0.000000
9,2011,OMITE_MAIOR_1,66,65,1,924.818462,131.506810,793.311651,ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO__NAO_REGRA_DE_EXCLUSAO,878.190159,-84.878508


Figura preservada em: outputs\figures\interactive\d18_05_influencia_pre.html


`ANALISE_DE_INFLUENCIA_PRE_TRATAMENTO` — `NAO_REGRA_DE_EXCLUSAO`. As omissões atenuam fortemente algumas diferenças, especialmente 2009 e 2012, mas não eliminam a divergência em 2011–2012. O padrão combina concentração em municípios grandes com heterogeneidade mais difusa. A amostra original permanece intacta.

## 7. Coorte 2013: precisão, suporte e trajetória

In [9]:
traj_2013 = amostra.loc[(amostra['papel_causal'].eq(d18.PAPEL_TRATADO)) & (amostra['coorte_g'].eq(2013)) & (amostra['ano'].lt(2013)), ['codigo_municipio_ibge', 'municipio_fonte', 'uf_codigo', 'ano', d18.OUTCOME_PRIMARIO]].copy()
suporte_2013 = scores_att.loc[scores_att['tratado'] & scores_att['coorte_g'].eq(2013), ['codigo_municipio_ibge', 'propensity_score', 'status_suporte_att', 'macrorregiao']]
resumo_2013 = traj_2013.groupby(['codigo_municipio_ibge', 'municipio_fonte', 'uf_codigo'], as_index=False).apply(lambda x: pd.Series({'slope_nivel_pre': np.polyfit(x['ano'], x[d18.OUTCOME_PRIMARIO], 1)[0], 'slope_log_pre': np.polyfit(x['ano'], np.log1p(x[d18.OUTCOME_PRIMARIO]), 1)[0], 'min_pre': x[d18.OUTCOME_PRIMARIO].min(), 'max_pre': x[d18.OUTCOME_PRIMARIO].max()}), include_groups=False).reset_index(drop=True).merge(suporte_2013, on='codigo_municipio_ibge', how='left', validate='one_to_one')
mostrar(resumo_2013, 10)
fig = px.line(traj_2013, x='ano', y=d18.OUTCOME_PRIMARIO, color='municipio_fonte', markers=True, labels={'ano': 'Ano pré-tratamento', d18.OUTCOME_PRIMARIO: 'Pessoal ocupado assalariado', 'municipio_fonte': 'Município'}, title='Coorte 2013: trajetórias individuais antes de g')
salvar_figura(fig, 'd18_06_coorte_2013_trajetorias')
fig.show()

,codigo_municipio_ibge,municipio_fonte,uf_codigo,slope_nivel_pre,slope_log_pre,min_pre,max_pre,propensity_score,status_suporte_att,macrorregiao
0,3300100,Angra dos Reis - RJ,33,3384.657143,0.098779,27165.000000,43488.000000,0.100949,TRATADO_DENTRO_FAIXA_CONTROLES,Sudeste
1,3542602,Registro - SP,35,822.885714,0.083352,8023.000000,12067.000000,0.041811,TRATADO_DENTRO_FAIXA_CONTROLES,Sudeste


Figura preservada em: outputs\figures\interactive\d18_06_coorte_2013_trajetorias.html


Ambos os municípios de 2013 estão dentro da faixa empírica dos controles e pertencem ao Sudeste. Registro/SP tem trajetória crescente relativamente regular; Angra dos Reis/RJ é muito mais volátil. Classificação: problema principal `PRECISAO`, risco secundário `TRAJETORIA`, sem evidência de `SUPORTE` insuficiente neste score. A coorte não é excluída.

## 8. Spillover: cenários hipotéticos de controles

In [10]:
spillover = d18.cenarios_spillover_hipoteticos(amostra, distancias, arranjos)
spillover.to_csv(OUT_DIAG / 'D18_spillover_cenarios.csv', index=False, encoding='utf-8-sig')
spillover_total = spillover.loc[spillover['macrorregiao'].eq('TOTAL')].drop_duplicates(['cenario'])
mostrar(spillover_total[['cenario', 'n_controles_original', 'n_potencialmente_contaminados', 'n_remanescentes']], 10)
fig = px.bar(spillover_total, x='cenario', y='n_remanescentes', text='n_remanescentes', color_discrete_sequence=[CORES_IPT['AZUL_PRINCIPAL']], labels={'cenario': 'Cenário hipotético', 'n_remanescentes': 'Controles remanescentes'}, title='Volume de controles sob sensibilidades espaciais hipotéticas')
fig.update_traces(textposition='outside')
salvar_figura(fig, 'd18_07_spillover_remanescentes')
fig.show()

,cenario,n_controles_original,n_potencialmente_contaminados,n_remanescentes
0,ATE_25_KM,4963,374,4589
30,ATE_50_KM,4963,1475,3488
60,ATE_100_KM,4963,3640,1323
90,MESMO_ARRANJO_POPULACIONAL,4963,145,4818


Figura preservada em: outputs\figures\interactive\d18_07_spillover_remanescentes.html


`CENARIO_HIPOTETICO_DE_SENSIBILIDADE` — `NAO_AMOSTRA_PRINCIPAL`. Restariam 4.589/3.488/1.323 controles sob 25/50/100 km e 4.818 após a flag de arranjo populacional. Há volume substancial para uma robustez futura, inclusive no cenário de 100 km; isso não prova ausência de contaminação nem escolhe um raio.

## 9. Antecipaçao: fato, hipótese e diagnóstico pré

In [11]:
antecipacao = d18.diagnostico_antecipacao_pre(amostra)
mostrar(antecipacao, 30)
display(Markdown('**FATO:** 128/129 timings são `proxy_censo`.  **HIPOTESE_CONGELADA:** `anticipation=0`.  **RISCO:** mudanças podem anteceder o primeiro registro.'))

,grupo,tempo_relativo_pre,n_unidades,media_nivel,mediana_nivel,media_indice_g_menos_1,mediana_indice_g_menos_1,coorte_g,fato_timing,hipotese_congelada,rotulo_inferencia
0,CONTROLES_NEVER_TREATED,-2,4963,2246.382430,638.000000,114.965661,97.379287,2009,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
1,TRATADOS_COORTE,-2,21,19857.190476,14627.000000,99.761274,98.290963,2009,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
2,CONTROLES_NEVER_TREATED,-3,4963,2246.382430,638.000000,96.684100,90.799516,2010,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
3,CONTROLES_NEVER_TREATED,-2,4963,2322.327826,644.000000,94.386402,93.568803,2010,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
4,TRATADOS_COORTE,-3,27,11842.814815,7041.000000,93.109670,89.958060,2010,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
5,TRATADOS_COORTE,-2,27,12032.777778,7520.000000,99.957761,96.378256,2010,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
6,CONTROLES_NEVER_TREATED,-3,4963,2322.327826,644.000000,88.599876,88.455538,2011,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
7,CONTROLES_NEVER_TREATED,-2,4963,2449.761032,702.000000,96.109993,94.810971,2011,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
8,TRATADOS_COORTE,-3,66,15277.939394,7247.500000,84.066825,86.841219,2011,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO
9,TRATADOS_COORTE,-2,66,16013.954545,8444.000000,90.341033,92.839268,2011,128_DE_129_TIMINGS_PROXY_CENSO,ANTICIPATION_0_PERIODOS,DIAGNOSTICO_PRE_NAO_REDEFINE_G_NEM_ANTECIPACAO


**FATO:** 128/129 timings são `proxy_censo`.  **HIPOTESE_CONGELADA:** `anticipation=0`.  **RISCO:** mudanças podem anteceder o primeiro registro.

Os padrões em `k=-2` e `k=-3` não são uniformes entre coortes e escalas. Eles justificam manter a antecipação como risco e pré-especificar sensibilidade futura, mas não permitem redefinir `g` nem inventar nova janela nesta etapa.

## 10. Matriz de riscos refinada

In [12]:
riscos = pd.DataFrame([
 ['DIFERENCAS_DE_NIVEL_ESCALA', 'SMDs baseline altos; slopes em log/percentual muito menores', 'SIM', 'SIM', 'INCERTO', 'SIM', 'NAO', 'INCERTO'],
 ['TRATADOS_ACIMA_SUPORTE_CONTROLES', '5 tratados acima do score máximo dos controles', 'SIM', 'SIM', 'SIM', 'SIM', 'NAO', 'INCERTO'],
 ['CONTROLES_ABAIXO_FAIXA_TRATADOS', '1.361 controles abaixo do score mínimo dos tratados', 'SIM', 'NAO', 'NAO', 'NAO', 'NAO', 'NAO'],
 ['PRE_TENDENCIAS_HETEROGENEAS', 'Diferenças persistem em partes de 2011–2013 após normalização', 'SIM', 'SIM', 'INCERTO', 'SIM', 'NAO', 'INCERTO'],
 ['COORTE_2013_N_2', 'Dois tratados; ambos dentro da faixa de score', 'SIM', 'NAO', 'NAO', 'SIM', 'NAO', 'NAO'],
 ['SPILLOVER', 'Proximidade/arranjo observados; contaminação não comprovada', 'SIM', 'SIM', 'SIM', 'INCERTO', 'NAO', 'INCERTO'],
 ['ANTECIPACAO_TIMING_PROXY', '128/129 timings proxy; padrões pré não uniformes', 'SIM', 'SIM', 'SIM', 'INCERTO', 'NAO', 'INCERTO'],
 ['COVARIAVEIS_EXTERNAS_AUSENTES', 'Main D14 é incondicional; confundimento condicional segue aberto', 'SIM', 'SIM', 'INCERTO', 'INCERTO', 'NAO', 'INCERTO'],
 ['INFLUENCIA_MUNICIPIOS_GRANDES', 'Omissões diagnósticas atenuam, mas não zeram 2011–2012', 'SIM', 'SIM', 'INCERTO', 'SIM', 'NAO', 'INCERTO'],
], columns=['RISCO', 'EVIDENCIA_RESUMIDA', 'EVIDENCIA_OBSERVADA', 'PLAUSIBILIDADE_TEORICA', 'IMPACTO_POTENCIAL_IDENTIFICACAO', 'IMPACTO_POTENCIAL_PRECISAO', 'BLOQUEIA_IMPLEMENTACAO_TECNICA', 'BLOQUEIA_INTERPRETACAO_CAUSAL'])
riscos.to_csv(OUT_DIAG / 'D18_matriz_riscos.csv', index=False, encoding='utf-8-sig')
mostrar(riscos, 20)

,RISCO,EVIDENCIA_RESUMIDA,EVIDENCIA_OBSERVADA,PLAUSIBILIDADE_TEORICA,IMPACTO_POTENCIAL_IDENTIFICACAO,IMPACTO_POTENCIAL_PRECISAO,BLOQUEIA_IMPLEMENTACAO_TECNICA,BLOQUEIA_INTERPRETACAO_CAUSAL
0,DIFERENCAS_DE_NIVEL_ESCALA,SMDs baseline altos; slopes em log/percentual muito menores,SIM,SIM,INCERTO,SIM,NAO,INCERTO
1,TRATADOS_ACIMA_SUPORTE_CONTROLES,5 tratados acima do score máximo dos controles,SIM,SIM,SIM,SIM,NAO,INCERTO
2,CONTROLES_ABAIXO_FAIXA_TRATADOS,1.361 controles abaixo do score mínimo dos tratados,SIM,NAO,NAO,NAO,NAO,NAO
3,PRE_TENDENCIAS_HETEROGENEAS,Diferenças persistem em partes de 2011–2013 após normalização,SIM,SIM,INCERTO,SIM,NAO,INCERTO
4,COORTE_2013_N_2,Dois tratados; ambos dentro da faixa de score,SIM,NAO,NAO,SIM,NAO,NAO
5,SPILLOVER,Proximidade/arranjo observados; contaminação não comprovada,SIM,SIM,SIM,INCERTO,NAO,INCERTO
6,ANTECIPACAO_TIMING_PROXY,128/129 timings proxy; padrões pré não uniformes,SIM,SIM,SIM,INCERTO,NAO,INCERTO
7,COVARIAVEIS_EXTERNAS_AUSENTES,Main D14 é incondicional; confundimento condicional segue aberto,SIM,SIM,INCERTO,INCERTO,NAO,INCERTO
8,INFLUENCIA_MUNICIPIOS_GRANDES,"Omissões diagnósticas atenuam, mas não zeram 2011–2012",SIM,SIM,INCERTO,SIM,NAO,INCERTO


## 11. Plano futuro de robustez pré-especificado

In [13]:
plano_robustez = d18.plano_robustez_pre_especificado()
plano_robustez.to_csv(OUT_DIAG / 'D18_plano_robustez.csv', index=False, encoding='utf-8-sig')
mostrar(plano_robustez, 20)

,item,justificativa_ex_ante,status_pre_especificacao,uso_futuro
0,PRINCIPAL_NIVEL,Outcome primário D14 em nível,CONGELADA,FUTURA_ESTIMACAO
1,SENSIBILIDADE_LOG1P,Assimetria e escala observadas no baseline/pré,PRE_ESPECIFICADA_D14,FUTURA_ESTIMACAO
2,JANELA_K_MENOS3_A_MAIS2,Avaliar lead adicional com coortes 2010-2013,PRE_ESPECIFICADA_D14,FUTURA_ESTIMACAO
3,AGREGACAO_POR_COORTE,Heterogeneidade e tamanhos de coorte distintos,JUSTIFICATIVA_METODOLOGICA,FUTURA_ESTIMACAO
4,COORTE_2013,n=2 implica precisão muito limitada,JUSTIFICATIVA_PRE_TRATAMENTO,DIAGNOSTICO_E_REPORTE_SEPARADO
5,SPILLOVER_25_KM,Cenário espacial documentado antes da estimação,PRE_ESPECIFICADA_D14,FUTURA_SENSIBILIDADE
6,SPILLOVER_50_KM,Cenário espacial documentado antes da estimação,PRE_ESPECIFICADA_D14,FUTURA_SENSIBILIDADE
7,SPILLOVER_100_KM,Cenário espacial documentado antes da estimação,PRE_ESPECIFICADA_D14,FUTURA_SENSIBILIDADE
8,ARRANJO_POPULACIONAL_2010,Integração funcional potencial e limitação temporal conhecida,PRE_ESPECIFICADA_D14,FUTURA_SENSIBILIDADE
9,COVARIAVEIS_BASELINE_EXISTENTES,Ajuste apenas com medidas disponíveis antes do tratamento,CONDICIONAL_A_METODO_E_OVERLAP,FUTURA_SENSIBILIDADE


O plano é futuro e congelável. Cada item decorre do D14, dos diagnósticos pré-tratamento ou da literatura metodológica; nenhum foi escolhido por resultado pós-tratamento. Nenhuma estimação do plano é executada na D18.

## 12. Dois gates distintos e preservação final

In [14]:
hash_d15_depois = d17.sha256_arquivo(PATH_AMOSTRA)
assert hash_d15_antes == hash_d15_depois
assert set(amostra.loc[amostra['papel_causal'].eq(d18.PAPEL_TRATADO), 'coorte_g'].dropna().astype(int).unique()) == set(d18.COORTES)
cabo = amostra.loc[amostra['codigo_municipio_ibge'].astype('string').str.zfill(7).eq('3300704')]
assert cabo.loc[~cabo['elegivel_estimacao_principal'], 'ano'].tolist() == [2009]
GATES_D18 = {
 'D18_REAVALIACAO_PRE_ESTIMACAO_CONCLUIDA': 'SIM',
 'D18_OVERLAP_ATT_DOCUMENTADO': 'SIM',
 'D18_PRE_TENDENCIAS_ESCALA_AUDITADAS': 'SIM',
 'D18_INFLUENCIA_PRE_TRATAMENTO_AUDITADA': 'SIM',
 'D18_PLANO_ROBUSTEZ_PRE_ESPECIFICADO': 'SIM',
 'D18_PRONTO_TECNICAMENTE_PARA_IMPLEMENTAR_ESTIMADOR': 'SIM',
 'D18_IDENTIFICACAO_SUFICIENTE_PARA_INTERPRETACAO_CAUSAL': 'COM_RESSALVAS',
 'DESENHO_CAUSAL_APROVADO': 'NAO',
}
display(pd.DataFrame({'gate': GATES_D18.keys(), 'valor': GATES_D18.values()}))
display(pd.DataFrame([{'artefato': str(PATH_AMOSTRA.relative_to(ROOT)), 'sha256_antes': hash_d15_antes, 'sha256_depois': hash_d15_depois, 'byte_identical': hash_d15_antes == hash_d15_depois}]))

,gate,valor
0,D18_REAVALIACAO_PRE_ESTIMACAO_CONCLUIDA,SIM
1,D18_OVERLAP_ATT_DOCUMENTADO,SIM
2,D18_PRE_TENDENCIAS_ESCALA_AUDITADAS,SIM
3,D18_INFLUENCIA_PRE_TRATAMENTO_AUDITADA,SIM
4,D18_PLANO_ROBUSTEZ_PRE_ESPECIFICADO,SIM
5,D18_PRONTO_TECNICAMENTE_PARA_IMPLEMENTAR_ESTIM...,SIM
6,D18_IDENTIFICACAO_SUFICIENTE_PARA_INTERPRETACA...,COM_RESSALVAS
7,DESENHO_CAUSAL_APROVADO,NAO


,artefato,sha256_antes,sha256_depois,byte_identical
0,data\processed\amostra_causal_cempre_2007_2019...,7c24c01b569ed6d10f773d00599b5010e1607696f46d71...,7c24c01b569ed6d10f773d00599b5010e1607696f46d71...,True


## Takeaways

### Prontidão técnica

A população, o painel, o outcome, os papéis causais e o timing estão materializados e validados. Nenhum alerta D17/D18 impede escrever e testar, em etapa posterior, a implementação do estimador moderno para adoção escalonada. Portanto, o gate técnico muda de `NAO` na D17 para `SIM` na D18 — refinamento conceitual, não alteração da amostra.

### Identificação

A evidência não autoriza interpretação causal irrestrita. Há cinco tratados sem suporte empírico intervalar dos controles no score diagnóstico, heterogeneidade pré residual, influência de porte, timing majoritariamente proxy e spillover plausível. Ao mesmo tempo, diferenças de nível e caudas de controles não são bloqueadores automáticos; a coorte 2013 é primariamente um problema de precisão. O gate é `COM_RESSALVAS`, não `SIM` irrestrito nem rejeição definitiva.

### Decisões deliberadamente não tomadas

Nenhum efeito causal foi estimado; nenhuma unidade/coorte foi excluída; nenhum trimming, matching, peso ou raio foi aplicado; `g`, tratamento, outcome, comparação principal e `anticipation=0` não foram alterados; o desenho causal não foi aprovado.